# Real World Data Wrangling Demo Notebook

This notebook is a live-demo companion for the Real World Data Wrangling presentation. It uses two small synthetic US public-library datasets so the demo can intentionally show common gathering, assessment, cleaning, merging, reshaping, missing-data, outlier, and visualization patterns.

The source starter notebook remains unchanged. This is the working copy.

## Demo Topic Map

Use this notebook as you present the project-flow portion of the deck:

- pandas refresher: import pandas, load CSV files, select Series, filter rows, and update values with `.loc`
- Gather: load the two local datasets that are used throughout the rest of the notebook
- Assess: `.head()`, `.tail()`, `.info()`, `.describe()`, `.value_counts()`, missing values, incorrect data types, duplicates, outliers, and tidiness issues
- Clean: copy DataFrames, standardize missing values, convert types, parse dates, filter known ranges, drop unknown-range outliers, handle duplicates, merge, concatenate rows, pivot, melt, transpose, group, aggregate, and resample time-series data
- Analyze: simple summaries and optional seaborn charts
- Advanced: optional scikit-learn imputation example

Standalone gathering demos for ZIP, JSON, API, and Kaggle are now in `Data_Gathering_Demos.ipynb`. Web scraping is intentionally excluded.


## 0. Setup

Keep all files in the same directory as this notebook so the relative paths work in the Udacity workspace and in local Jupyter.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(".")
programs_path = DATA_DIR / "library_program_registrations_dirty.csv"
branches_path = DATA_DIR / "library_branch_lookup.csv"

print("Program dataset exists:", programs_path.exists())
print("Branch dataset exists:", branches_path.exists())


Program dataset exists: True
Branch dataset exists: True


## 1. Gather Data

The project asks learners to gather from at least two sources. This notebook uses two local CSV sources that are both used in the later assessment, cleaning, and analysis sections:

1. `library_program_registrations_dirty.csv`: messy learner registration and attendance records.
2. `library_branch_lookup.csv`: branch metadata that can be merged into the registration data.

Use `Data_Gathering_Demos.ipynb` for standalone examples of ZIP files, JSON files, API requests, and Kaggle downloads.


In [2]:
# Load CSV files into pandas DataFrames.
programs_raw = pd.read_csv(programs_path)
branches_raw = pd.read_csv(branches_path)

programs_raw.head()

,registration_id,learner_name,Age,city,branch_id,program_id,program_name,session_date,program_type,interests,registration_status,attendance_hours,pre_score,post_score,fee_paid,scholarship_amount,empty_column
0,R001,Emma Johnson,8,New York,B101,P001,Intro to Python,2026-01-12,STEM,coding; robots,Attended,2.0,45,72,$0,$25,NaN
1,R002,Liam Smith,10,Los Angeles,B102,P001,Intro to Python,01/20/2026,STEM,coding; games,Attended,2.5,50,80,$0,$25,NaN
2,R003,Olivia Brown,11,Chicago,B103,P002,Data Storytelling,Feb 02 2026,Data,charts; storytelling,No Show,0,60,*,$5,$0,NaN
3,R004,Noah Davis,7,Houston,B104,P003,Robotics Basics,2026/02/15,STEM,robots; teamwork,attended,1.5,40,65,Free,$30,NaN
4,R005,Ava Wilson,16,Phoenix,B105,P004,College Essay Lab,2026-03-05,Writing,writing; college,Attended,2.0,70,88,$10,NaN,NaN


In [3]:
branches_raw.head()

,branch_id,branch_name,branch_city,state,region,opened_year,square_feet,annual_visits,wifi_available
0,B101,Central Library,New York,NY,Northeast,1911,80000,"1,250,000",True
1,B102,Westside Branch,Los Angeles,CA,West,1954,45000,"850,000",True
2,B103,Lakeview Branch,Chicago,IL,Midwest,1972,38000,"620,000",True
3,B104,Bayou Branch,Houston,TX,South,1985,32000,"540,000",False
4,B105,Desert Ridge Branch,Phoenix,AZ,West,2001,28000,"410,000",True


### pandas Refresher: DataFrame and Series

A DataFrame is a table. A Series is one column. The bracket style is the safest way to select a column because it works even when names have spaces or special characters.

In [4]:
# Get one column as a Series.
ages = programs_raw["Age"]
ages.head()

0     8
1    10
2    11
3     7
4    16
Name: Age, dtype: object

In [5]:
# Dot notation can work for simple column names, but bracket notation is safer.
program_names = programs_raw.program_name
program_names.head()

0      Intro to Python
1      Intro to Python
2    Data Storytelling
3      Robotics Basics
4    College Essay Lab
Name: program_name, dtype: object

In [6]:
# Select rows where Age is less than 11.
# We convert first because this dirty column contains values like "unknown".
age_numeric_preview = pd.to_numeric(programs_raw["Age"], errors="coerce")
children_preview = programs_raw[age_numeric_preview < 11]
children_preview[["learner_name", "Age", "program_name"]]

,learner_name,Age,program_name
0,Emma Johnson,8,Intro to Python
1,Liam Smith,10,Intro to Python
3,Noah Davis,7,Robotics Basics
6,Mia Garcia,9,Robotics Basics
15,Evelyn Lewis,-1,Chess Club
19,Abigail Young,10,Game Design
20,Henry King,8,Game Design


In [7]:
# Update a status column with .loc.
programs_preview = programs_raw.copy()
programs_preview.loc[age_numeric_preview < 11, "learner_group"] = "Children"
programs_preview.loc[age_numeric_preview >= 11, "learner_group"] = "Teen or Adult"
programs_preview[["learner_name", "Age", "learner_group"]].head(10)

,learner_name,Age,learner_group
0,Emma Johnson,8,Children
1,Liam Smith,10,Children
2,Olivia Brown,11,Teen or Adult
3,Noah Davis,7,Children
4,Ava Wilson,16,Teen or Adult
5,James Miller,35,Teen or Adult
6,Mia Garcia,9,Children
7,William Martinez,42,Teen or Adult
8,Sophia Anderson,14,Teen or Adult
9,Benjamin Thomas,13,Teen or Adult


## 2. Assess Data

Start broad, then get specific. The goal is to identify data quality issues and data tidiness issues before cleaning.

In [12]:
programs_raw.head()

,registration_id,learner_name,Age,city,branch_id,program_id,program_name,session_date,program_type,interests,registration_status,attendance_hours,pre_score,post_score,fee_paid,scholarship_amount,empty_column
0,R001,Emma Johnson,8,New York,B101,P001,Intro to Python,2026-01-12,STEM,coding; robots,Attended,2.0,45,72,$0,$25,NaN
1,R002,Liam Smith,10,Los Angeles,B102,P001,Intro to Python,01/20/2026,STEM,coding; games,Attended,2.5,50,80,$0,$25,NaN
2,R003,Olivia Brown,11,Chicago,B103,P002,Data Storytelling,Feb 02 2026,Data,charts; storytelling,No Show,0,60,*,$5,$0,NaN
3,R004,Noah Davis,7,Houston,B104,P003,Robotics Basics,2026/02/15,STEM,robots; teamwork,attended,1.5,40,65,Free,$30,NaN
4,R005,Ava Wilson,16,Phoenix,B105,P004,College Essay Lab,2026-03-05,Writing,writing; college,Attended,2.0,70,88,$10,NaN,NaN


In [13]:
programs_raw.tail()

,registration_id,learner_name,Age,city,branch_id,program_id,program_name,session_date,program_type,interests,registration_status,attendance_hours,pre_score,post_score,fee_paid,scholarship_amount,empty_column
23,R024,Grace Green,36,New York,B101,P015,Community Budgeting,2026-10-01,Finance,budgeting; community,Attended,2.0,105,87,$0,$10,NaN
24,R025,Daniel Baker,19,Los Angeles,B102,P016,Podcasting,2026-10-16,Media,audio; storytelling,NaN,2.0,59,80,$8,$0,NaN
25,R026,Ella Adams,NaN,Chicago,B103,P017,Intro to AI,2026-11-05,STEM,ai; ethics,Attended,2.0,52,85,$0,$25,NaN
26,R027,Matthew Nelson,22,Houston,B104,P017,Intro to AI,2026-11-12,STEM,ai; ethics,Attended,*,54,86,$0,$25,NaN
27,R028,Sarah Carter,44,Phoenix,B105,P018,Community Garden Data,2026-12-03,Environment,gardening; data,Attended,2.5,57,83,$5,$5,NaN


In [14]:
programs_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   registration_id      28 non-null     object 
 1   learner_name         28 non-null     object 
 2   Age                  27 non-null     object 
 3   city                 28 non-null     object 
 4   branch_id            28 non-null     object 
 5   program_id           28 non-null     object 
 6   program_name         28 non-null     object 
 7   session_date         28 non-null     object 
 8   program_type         28 non-null     object 
 9   interests            28 non-null     object 
 10  registration_status  27 non-null     object 
 11  attendance_hours     28 non-null     object 
 12  pre_score            28 non-null     object 
 13  post_score           28 non-null     object 
 14  fee_paid             28 non-null     object 
 15  scholarship_amount   27 non-null     objec

In [15]:
programs_raw.describe(include="all")

,registration_id,learner_name,Age,city,branch_id,program_id,program_name,session_date,program_type,interests,registration_status,attendance_hours,pre_score,post_score,fee_paid,scholarship_amount,empty_column
count,28,28,27,28,28,28,28,28,28,28,27,28,28,28,28,27,0.0
unique,28,27,24,8,8,18,18,27,12,20,5,9,25,23,8,6,NaN
top,R001,Benjamin Thomas,8,Los Angeles,B102,P004,College Essay Lab,2026-04-20,STEM,coding; games,Attended,2.0,50,80,$0,$0,NaN
freq,1,2,2,5,5,3,3,2,8,3,22,12,2,2,12,13,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
programs_raw["registration_status"].value_counts(dropna=False)

registration_status
Attended     22
No Show       2
attended      1
Cancelled     1
attended      1
NaN           1
Name: count, dtype: int64

In [17]:
branches_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   branch_id       7 non-null      object
 1   branch_name     7 non-null      object
 2   branch_city     7 non-null      object
 3   state           7 non-null      object
 4   region          7 non-null      object
 5   opened_year     7 non-null      int64 
 6   square_feet     7 non-null      int64 
 7   annual_visits   7 non-null      object
 8   wifi_available  7 non-null      bool  
dtypes: bool(1), int64(2), object(6)
memory usage: 587.0+ bytes


### Assessment Checks

Look for missing values, empty columns, incorrect data types, duplicate records, values outside plausible ranges, and columns that contain multiple values.

In [18]:
programs_raw.isna().sum().sort_values(ascending=False)

empty_column           28
registration_status     1
Age                     1
scholarship_amount      1
registration_id         0
program_id              0
learner_name            0
city                    0
branch_id               0
program_type            0
session_date            0
program_name            0
interests               0
pre_score               0
attendance_hours        0
fee_paid                0
post_score              0
dtype: int64

In [19]:
empty_columns = programs_raw.columns[programs_raw.isna().all()].tolist()
empty_columns

['empty_column']

In [20]:
missing_markers = ["TBD", "unknown", "*", "#", "N/A"]
marker_counts = {}
for column in programs_raw.columns:
    marker_counts[column] = programs_raw[column].isin(missing_markers).sum()

pd.Series(marker_counts).sort_values(ascending=False).head(10)

post_score          2
attendance_hours    2
Age                 1
pre_score           1
registration_id     0
program_id          0
learner_name        0
city                0
branch_id           0
program_type        0
dtype: int64

In [21]:
duplicate_mask = programs_raw.duplicated(
    subset=["learner_name", "program_id", "session_date"],
    keep=False,
)
programs_raw.loc[duplicate_mask, ["registration_id", "learner_name", "program_id", "session_date", "registration_status"]]

,registration_id,learner_name,program_id,session_date,registration_status
9,R010,Benjamin Thomas,P004,2026-04-20,Attended
10,R011,Benjamin Thomas,P004,2026-04-20,attended


In [22]:
age_as_number = pd.to_numeric(programs_raw["Age"], errors="coerce")
pre_score_as_number = pd.to_numeric(programs_raw["pre_score"], errors="coerce")

range_issue_rows = programs_raw[(age_as_number < 0) | (age_as_number > 100) | (pre_score_as_number > 100)]
range_issue_rows[["registration_id", "learner_name", "Age", "pre_score"]]

,registration_id,learner_name,Age,pre_score
11,R012,Charlotte Moore,130,50
15,R016,Evelyn Lewis,-1,35
23,R024,Grace Green,36,105


In [23]:
programs_raw[["registration_id", "interests"]].head(8)

,registration_id,interests
0,R001,coding; robots
1,R002,coding; games
2,R003,charts; storytelling
3,R004,robots; teamwork
4,R005,writing; college
5,R006,career; writing
6,R007,robots; coding
7,R008,spreadsheets; business


### Assessment Summary

Data quality issues found:

- `Age` has unknown, blank, negative, and implausibly high values.
- `attendance_hours` contains `TBD`, `*`, and an extreme value.
- `pre_score` and `post_score` include nonnumeric markers and an invalid score above 100.
- `registration_status` has inconsistent capitalization and a blank value.
- `empty_column` contains no useful values.
- One `branch_id` in the registrations is not in the branch lookup table.

Data tidiness issues found:

- `interests` stores multiple values in one column.
- Branch details live in a separate table and need to be merged with the registration table for analysis.


## 3. Clean Data

Always start by copying the raw DataFrames. This lets you reset without reloading files.

In [ ]:
programs_clean = programs_raw.copy()
branches_clean = branches_raw.copy()

### Standardize Missing Values and Text

Make missing markers consistent before converting data types.

In [ ]:
missing_marker_map = {
    "TBD": np.nan,
    "unknown": np.nan,
    "*": np.nan,
    "#": np.nan,
    "N/A": np.nan,
}
programs_clean = programs_clean.replace(missing_marker_map)
programs_clean = programs_clean.replace(r"^\s*$", np.nan, regex=True)

for column in programs_clean.select_dtypes(include="object").columns:
    programs_clean[column] = programs_clean[column].str.strip()

programs_clean["registration_status"] = programs_clean["registration_status"].str.title()
programs_clean["registration_status"] = programs_clean["registration_status"].fillna("Unknown")
programs_clean["registration_status"].value_counts(dropna=False)

### Convert Data Types

After missing values are standardized, convert numeric, money, date, and boolean columns.

In [ ]:
def money_to_number(series):
    cleaned = (
        series.astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"Free": "0", "free": "0", "<NA>": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_mixed_dates(series):
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")


numeric_columns = ["Age", "attendance_hours", "pre_score", "post_score"]
for column in numeric_columns:
    programs_clean[column] = pd.to_numeric(programs_clean[column], errors="coerce")

programs_clean["fee_paid"] = money_to_number(programs_clean["fee_paid"])
programs_clean["scholarship_amount"] = money_to_number(programs_clean["scholarship_amount"])
programs_clean["session_date"] = parse_mixed_dates(programs_clean["session_date"])

branches_clean["annual_visits"] = pd.to_numeric(
    branches_clean["annual_visits"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
branches_clean["wifi_available"] = branches_clean["wifi_available"].astype(bool)

programs_clean.dtypes

### Known Range: Filter Valid Values

When a valid range is known, use boolean selectors. Here, learner age should be 0 to 100 and scores should be 0 to 100.

In [ ]:
known_range_issues = programs_clean[
    (programs_clean["Age"] < 0)
    | (programs_clean["Age"] > 100)
    | (programs_clean["pre_score"] < 0)
    | (programs_clean["pre_score"] > 100)
    | (programs_clean["post_score"] < 0)
    | (programs_clean["post_score"] > 100)
]
known_range_issues[["registration_id", "learner_name", "Age", "pre_score", "post_score"]]

In [ ]:
programs_clean = programs_clean.drop(index=known_range_issues.index)
print("Rows after dropping known-range issues:", len(programs_clean))

### Missing Data: Remove or Impute?

Compare what happens if we remove rows with missing attendance versus imputing attendance with the program-type mean.

In [ ]:
attendance_original = programs_clean["attendance_hours"].describe()
attendance_after_removal = programs_clean.dropna(subset=["attendance_hours"])["attendance_hours"].describe()

programs_imputed = programs_clean.copy()
programs_imputed["attendance_hours"] = programs_imputed.groupby("program_type")["attendance_hours"].transform(
    lambda values: values.fillna(values.mean())
)
attendance_after_imputation = programs_imputed["attendance_hours"].describe()

pd.DataFrame({
    "original": attendance_original,
    "after_removal": attendance_after_removal,
    "after_imputation": attendance_after_imputation,
})

### Unknown Range: Drop Outliers with Mean and Standard Deviation

When the valid range is not known, a simple first pass is to inspect values above mean plus a chosen number of standard deviations. Here, attendance of 900 hours is clearly unrealistic for one library session.

In [ ]:
attendance_mean = programs_imputed["attendance_hours"].mean()
attendance_std = programs_imputed["attendance_hours"].std()
upper_bound = attendance_mean + attendance_std

attendance_outliers = programs_imputed[programs_imputed["attendance_hours"] > upper_bound]
print("Upper bound:", upper_bound)
attendance_outliers[["registration_id", "learner_name", "attendance_hours"]]

In [ ]:
programs_imputed = programs_imputed.drop(index=attendance_outliers.index)
print("Rows after dropping attendance outliers:", len(programs_imputed))

### Duplicates

For this demo, treat rows with the same learner, program, and parsed session date as duplicate registrations.

In [ ]:
duplicate_mask = programs_imputed.duplicated(
    subset=["learner_name", "program_id", "session_date"],
    keep=False,
)
programs_imputed.loc[duplicate_mask, ["registration_id", "learner_name", "program_id", "session_date"]]

In [ ]:
programs_deduped = programs_imputed.drop_duplicates(
    subset=["learner_name", "program_id", "session_date"],
    keep="last",
)
print("Rows after dropping duplicates:", len(programs_deduped))

### Use `.loc` for Targeted Updates

This mirrors the pandas refresher slide: update one column for rows that match a condition.

In [ ]:
programs_deduped = programs_deduped.copy()
programs_deduped.loc[programs_deduped["Age"] < 11, "learner_group"] = "Children"
programs_deduped.loc[programs_deduped["Age"].between(11, 17), "learner_group"] = "Teen"
programs_deduped.loc[programs_deduped["Age"] >= 18, "learner_group"] = "Adult"
programs_deduped["learner_group"].value_counts(dropna=False)

### Merge DataFrames

The registration records and branch lookup are two observational units. Merge them when you need a single analysis table.

In [ ]:
merged = pd.merge(programs_deduped, branches_clean, on="branch_id", how="left", indicator=True)
merged[["_merge", "branch_id"]].value_counts(dropna=False)

In [ ]:
merged[merged["_merge"] != "both"][["registration_id", "learner_name", "branch_id", "program_name"]]

### Append Rows with `pd.concat()`

Modern pandas uses `pd.concat()` instead of the old `.append()` method.

In [ ]:
new_registration = pd.DataFrame([
    {
        "registration_id": "R029",
        "learner_name": "Nora Mitchell",
        "Age": 31,
        "city": "New York",
        "branch_id": "B101",
        "program_id": "P019",
        "program_name": "Data Cleaning Practice",
        "session_date": pd.Timestamp("2026-12-10"),
        "program_type": "Data",
        "interests": "cleaning; pandas",
        "registration_status": "Attended",
        "attendance_hours": 2.0,
        "pre_score": 61,
        "post_score": 86,
        "fee_paid": 0,
        "scholarship_amount": 10,
        "empty_column": np.nan,
        "learner_group": "Adult",
    }
])

programs_appended = pd.concat([programs_deduped, new_registration], ignore_index=True)
print("Before:", len(programs_deduped), "After:", len(programs_appended))

### Tidy Data: Split and Explode Multi-Value Columns

The `interests` column contains multiple values in one cell. Split it into a list and then explode to one interest per row.

In [ ]:
interests_long = (
    programs_deduped
    .assign(interest=programs_deduped["interests"].str.split(";"))
    .explode("interest")
)
interests_long["interest"] = interests_long["interest"].str.strip()
interests_long[["registration_id", "learner_name", "interest"]].head(12)

### Pivot and Melt

`melt()` makes wide data longer. `pivot()` can spread it back out.

In [ ]:
score_long = programs_deduped.melt(
    id_vars=["registration_id", "learner_name", "program_type"],
    value_vars=["pre_score", "post_score"],
    var_name="assessment",
    value_name="score",
)
score_long.head(8)

In [ ]:
score_wide = score_long.pivot(
    index=["registration_id", "learner_name", "program_type"],
    columns="assessment",
    values="score",
).reset_index()
score_wide.head()

### Transposition

`.T` transposes rows and columns. It is an attribute, not a function, so do not add parentheses.

In [ ]:
programs_deduped[["registration_id", "learner_name", "Age", "attendance_hours"]].head(4).T

### Grouping and Aggregation

Group by categories, then calculate summaries such as count, mean, and sum.

In [ ]:
program_summary = merged.groupby("program_type").agg(
    registrations=("registration_id", "count"),
    average_age=("Age", "mean"),
    average_attendance_hours=("attendance_hours", "mean"),
    total_fees=("fee_paid", "sum"),
    average_post_score=("post_score", "mean"),
).sort_values("registrations", ascending=False)
program_summary

In [ ]:
branch_summary = merged.groupby(["region", "branch_city"]).agg(
    registrations=("registration_id", "count"),
    average_attendance_hours=("attendance_hours", "mean"),
    total_scholarship=("scholarship_amount", "sum"),
).sort_values("registrations", ascending=False)
branch_summary

### Periodic Means with `.resample()`

When the data has dates, use resampling to compute period-level means. This is useful for time-series analysis and for period-based imputation.

In [ ]:
time_indexed = programs_deduped.dropna(subset=["session_date"]).set_index("session_date").sort_index()
monthly_attendance = time_indexed.resample("ME")["attendance_hours"].mean()
monthly_attendance

In [ ]:
programs_with_monthly_mean = programs_deduped.copy()
programs_with_monthly_mean["session_month"] = programs_with_monthly_mean["session_date"].dt.to_period("M")
monthly_lookup = programs_with_monthly_mean.groupby("session_month")["attendance_hours"].mean()
programs_with_monthly_mean["monthly_mean_attendance"] = programs_with_monthly_mean["session_month"].map(monthly_lookup)
programs_with_monthly_mean[["registration_id", "session_date", "attendance_hours", "monthly_mean_attendance"]].head()

### Optional Advanced Preprocessing with Scikit-Learn

This mirrors the advanced preprocessing slide. It is optional for this project, so the cell skips gracefully when scikit-learn is not installed.

In [ ]:
try:
    from sklearn.impute import SimpleImputer

    sklearn_demo = programs_deduped[["Age", "attendance_hours", "pre_score", "post_score"]].copy()
    simple_imp = SimpleImputer(missing_values=np.nan, strategy="mean")
    sklearn_demo[:] = simple_imp.fit_transform(sklearn_demo)
    print("Scikit-learn imputation completed.")
    sklearn_demo.head()
except ImportError:
    print("Optional dependency missing. To run this section: python -m pip install scikit-learn")

## 4. Analyze and Visualize

Keep the final analysis simple. The goal is to show learners how cleaned data supports clearer questions.

In [ ]:
analysis_table = merged[merged["_merge"] == "both"].copy()
analysis_table["score_gain"] = analysis_table["post_score"] - analysis_table["pre_score"]

analysis_table.groupby("program_type").agg(
    registrations=("registration_id", "count"),
    average_score_gain=("score_gain", "mean"),
    average_attendance_hours=("attendance_hours", "mean"),
).sort_values("average_score_gain", ascending=False)

In [ ]:
try:
    import seaborn as sns
    import matplotlib.pyplot as plt

    sns.histplot(data=analysis_table, x="Age", bins=8)
    plt.title("Learner Ages")
    plt.show()

    sns.relplot(data=analysis_table, x="pre_score", y="post_score", hue="program_type")
    plt.title("Pre Score vs Post Score")
    plt.show()
except ImportError:
    print("Optional dependencies missing. To run charts: python -m pip install seaborn matplotlib")

## 5. Save the Final Dataset

This mirrors the project submission requirement: preserve raw data, keep the cleaning notebook, and export a final cleaned dataset.

In [ ]:
final_columns = [
    "registration_id", "learner_name", "Age", "learner_group", "city",
    "branch_id", "branch_name", "branch_city", "state", "region",
    "program_id", "program_name", "program_type", "session_date",
    "registration_status", "attendance_hours", "pre_score", "post_score",
    "fee_paid", "scholarship_amount", "interests",
]

final_dataset = analysis_table[final_columns].copy()
final_path = DATA_DIR / "library_programs_cleaned.csv"
final_dataset.to_csv(final_path, index=False)
print("Saved final cleaned dataset to:", final_path)
print("Final shape:", final_dataset.shape)

## Teaching Prompts

Use these during the live session:

- Before assessment: "What problems do you expect to find in real-world registration data?"
- After `.info()`: "Which columns have suspicious data types?"
- Before cleaning outliers: "Should we remove this value automatically, or investigate first?"
- Before merge: "What should happen to a registration whose branch_id does not appear in the branch table?"
- Before visualizations: "Which chart would help us compare learning gains across program types?"